# Model Training

Training script for CompCars classification experiments.

This notebook trains multiple models with different architectures and loss functions:
- **Architectures**: SimpleCNN, ResNet50, EfficientNet-B0, ResNet50-SE, Hierarchical
- **Loss Functions**: Cross-Entropy, Label Smoothing, Focal Loss, Hierarchical Loss

If you re-downloaded the dataset, remember to put it in `/dataset` and re-run the `/src/preprocess.py` beforehand!


## 1. Setup

In [ ]:
# imports
import sys
from pathlib import Path
from datetime import datetime
import json

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from tqdm import tqdm

# add src to path
sys.path.insert(0, str(Path('.').absolute()))

from src.dataset import (
    ProcessedCompCarsDataset,
    get_processed_train_transforms,
    get_processed_val_transforms,
    get_dataloader
)
from src.models import (
    SimpleCNN,
    ResNet50Classifier,
    EfficientNetClassifier,
    ResNet50_SE,
    HierarchicalCarClassifier,
    HierarchicalLoss,
    count_parameters
)
from src.training import (
    TrainingHistory,
    EarlyStopping,
    train_one_epoch,
    train_one_epoch_with_accumulation,
    validate,
    plot_training_curves,
    plot_hierarchical_curves,
    save_checkpoint,
    get_optimizer_with_differential_lr,
    train_hierarchical_epoch,
    validate_hierarchical
)
from src.losses import get_loss_function
from src.utils import get_model, save_config, is_pretrained_model

## 2. Configuration

In [ ]:
# experiments to run: list of (model_name, loss_type) tuples
# available models: 'simplecnn', 'resnet50', 'efficientnet', 'resnet50_se', 'hierarchical'
# available losses: 'ce', 'label_smoothing', 'focal', 'weighted_ce', 'weighted_focal'

# full experiment matrix (7 experiments from strategic plan)
EXPERIMENTS = []
# loss function comparison on resnet50
EXPERIMENTS.append(('resnet50', 'label_smoothing'))
EXPERIMENTS.append(('resnet50', 'ce'))
EXPERIMENTS.append(('resnet50', 'focal'))
# architecture comparison (all use label_smoothing)
EXPERIMENTS.append(('simplecnn', 'label_smoothing'))
EXPERIMENTS.append(('efficientnet', 'label_smoothing'))
EXPERIMENTS.append(('resnet50_se', 'label_smoothing'))
# hierarchical (uses its own loss)
EXPERIMENTS.append(('hierarchical', 'hierarchical'))

# uncomment below to run a single experiment for testing
# EXPERIMENTS = [('resnet50', 'label_smoothing')]

In [3]:
# dataset configuration
# using pre-processed images (224x224, bbox-cropped)
PROCESSED_DATA_ROOT = 'dataset/processed'
TASK = 'make'

# class counts will be determined from dataset
# (75 makes, 431 models in processed dataset)

# hierarchical training configuration
HIERARCHICAL_ALPHA = 0.3  # weight for make loss (1-alpha for model loss)

In [4]:
# training configuration (shared by all models)
BATCH_SIZE = 32
NUM_EPOCHS = 100  # full training epochs
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4

# loss function configuration (defaults, can be overridden per experiment)
LABEL_SMOOTHING = 0.1
FOCAL_GAMMA = 2.0

# differential learning rates (for pretrained models)
USE_DIFFERENTIAL_LR = True
BACKBONE_LR = 0.0001  # lower lr for pretrained backbone
HEAD_LR = 0.001  # higher lr for new classifier head

# gradient accumulation (set > 1 if running out of VRAM)
ACCUMULATION_STEPS = 1  # effective batch = BATCH_SIZE * ACCUMULATION_STEPS

# scheduler configuration
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5

# early stopping
EARLY_STOPPING_PATIENCE = 100 # 5 before, set high to disable for better plotting

# mixed precision
USE_AMP = True

In [ ]:
# paths
CHECKPOINT_DIR = Path('checkpoints')
RESULTS_DIR = Path('results')

# random seed for reproducibility - my student id
SEED = 2128831

# per-model config overrides (optional)
# if a model needs different settings, add them here
MODEL_CONFIGS = {}
MODEL_CONFIGS['resnet50'] = {'learning_rate': 0.0001}  # lower lr for pretrained backbone
MODEL_CONFIGS['efficientnet'] = {'learning_rate': 0.0001}  # lower lr for pretrained backbone
MODEL_CONFIGS['resnet50_se'] = {'learning_rate': 0.0001}  # lower lr for pretrained backbone with se attention
MODEL_CONFIGS['hierarchical'] = {'learning_rate': 0.0001}  # lower lr for pretrained backbone

## 3. Helper Functions

In [ ]:
def get_config_for_model(model_name, loss_type, num_classes, num_models=None):
    # get configuration for a specific model and loss type
    # uses the shared function from src.utils with local constants
    from src.utils import get_config_for_model as utils_get_config
    
    config = utils_get_config(
        model_name=model_name,
        loss_type=loss_type,
        num_classes=num_classes,
        num_models=num_models,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        scheduler_patience=SCHEDULER_PATIENCE,
        scheduler_factor=SCHEDULER_FACTOR,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        label_smoothing=LABEL_SMOOTHING,
        focal_gamma=FOCAL_GAMMA,
        use_differential_lr=USE_DIFFERENTIAL_LR,
        backbone_lr=BACKBONE_LR,
        head_lr=HEAD_LR,
        accumulation_steps=ACCUMULATION_STEPS,
        use_amp=USE_AMP,
        seed=SEED,
        task=TASK,
        model_configs=MODEL_CONFIGS
    )
    
    return config

## 4. Standard Training Function

In [ ]:
def train_model(model_name, loss_type, train_loader, val_loader, device, num_classes, num_models=None):
    # train a single model with specified loss function

    # get config for this model
    config = get_config_for_model(model_name, loss_type, num_classes, num_models)

    # create experiment name (model + loss type for unique outputs)
    experiment_name = f"{model_name}_{loss_type}"

    print("\n" + "=" * 60)
    print(f"TRAINING: {experiment_name.upper()}")
    print("=" * 60)

    # create model
    print(f"\ncreating model: {model_name}")
    model = get_model(model_name, config['num_classes'])
    model = model.to(device)

    # print model info
    params = count_parameters(model)
    print(f"total parameters: {params['total']:,}")
    print(f"trainable parameters: {params['trainable']:,}")

    # loss function (using factory)
    criterion = get_loss_function(
        loss_type=config['loss_type'],
        num_classes=config['num_classes'],
        label_smoothing=config['label_smoothing'],
        focal_gamma=config['focal_gamma']
    )
    print(f"loss function: {config['loss_type']}")

    # optimizer with optional differential learning rates
    # use is_pretrained_model from utils
    use_diff_lr = config['use_differential_lr'] and is_pretrained_model(model_name)

    if use_diff_lr:
        optimizer = get_optimizer_with_differential_lr(
            model,
            backbone_lr=config['backbone_lr'],
            head_lr=config['head_lr'],
            weight_decay=config['weight_decay']
        )
        print(f"using differential lr: backbone={config['backbone_lr']}, head={config['head_lr']}")
    else:
        optimizer = optim.AdamW(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay']
        )

    # scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience']
    )

    # mixed precision scaler
    scaler = GradScaler('cuda')

    # early stopping
    early_stopping = EarlyStopping(patience=config['early_stopping_patience'])

    # training history
    history = TrainingHistory()

    # best model tracking
    best_val_acc = 0.0

    # save config
    config_path = CHECKPOINT_DIR / f'{experiment_name}_config.json'
    save_config(config, config_path)
    print(f"saved config to: {config_path}")

    # training loop
    print(f"\nstarting training for {config['num_epochs']} epochs...")
    print(f"batch size: {config['batch_size']}")
    if use_diff_lr:
        print(f"backbone lr: {config['backbone_lr']}, head lr: {config['head_lr']}")
    else:
        print(f"learning rate: {config['learning_rate']}")
    if config['accumulation_steps'] > 1:
        effective_batch = config['batch_size'] * config['accumulation_steps']
        print(f"gradient accumulation: {config['accumulation_steps']} steps (effective batch: {effective_batch})")
    print("-" * 50)

    start_time = datetime.now()

    for epoch in range(config['num_epochs']):
        print(f"\nepoch {epoch + 1}/{config['num_epochs']}")

        # get current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        # train (with optional gradient accumulation)
        if config['accumulation_steps'] > 1:
            train_loss, train_acc = train_one_epoch_with_accumulation(
                model, train_loader, optimizer, criterion, device, scaler,
                config['use_amp'], config['accumulation_steps']
            )
        else:
            train_loss, train_acc = train_one_epoch(
                model, train_loader, optimizer, criterion, device, scaler, config['use_amp']
            )

        # validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)

        # update history
        history.update(train_loss, train_acc, val_loss, val_acc, current_lr)

        # print epoch summary
        print(f"train loss: {train_loss:.4f} | train acc: {train_acc:.2f}%")
        print(f"val loss: {val_loss:.4f} | val acc: {val_acc:.2f}%")
        print(f"learning rate: {current_lr:.6f}")

        # update scheduler
        scheduler.step(val_acc)

        # save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            checkpoint_path = CHECKPOINT_DIR / f'{experiment_name}_best.pth'
            save_checkpoint(model, optimizer, epoch, val_acc, checkpoint_path)
            print(f"saved best model (val_acc: {val_acc:.2f}%)")

        # check early stopping
        if early_stopping(val_acc):
            print(f"\nearly stopping triggered at epoch {epoch + 1}")
            break

    end_time = datetime.now()
    training_time = end_time - start_time

    # save training history
    history_path = CHECKPOINT_DIR / f'{experiment_name}_history.json'
    history.save(history_path)
    print(f"\nsaved history to: {history_path}")

    # plot training curves
    plot_path = RESULTS_DIR / f'{experiment_name}_training_curves.png'
    plot_training_curves(history, save_path=plot_path)

    # save summary
    summary = {}
    summary['experiment_name'] = experiment_name
    summary['model_name'] = model_name
    summary['loss_type'] = config['loss_type']
    summary['total_parameters'] = params['total']
    summary['trainable_parameters'] = params['trainable']
    summary['epochs_trained'] = len(history.train_loss)
    summary['best_val_accuracy'] = best_val_acc
    summary['final_train_accuracy'] = history.train_acc[-1]
    summary['final_val_accuracy'] = history.val_acc[-1]
    summary['training_time'] = str(training_time)
    summary['checkpoint_path'] = str(CHECKPOINT_DIR / f'{experiment_name}_best.pth')

    summary_path = CHECKPOINT_DIR / f'{experiment_name}_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"saved summary to: {summary_path}")

    # print final summary
    print("\n" + "-" * 50)
    print(f"{experiment_name.upper()} COMPLETE")
    print(f"best val accuracy: {best_val_acc:.2f}%")
    print(f"training time: {training_time}")
    print("-" * 50)

    return best_val_acc, training_time

## 5. Hierarchical Training Functions

In [ ]:
def train_hierarchical_model(model_name, loss_type, train_loader, val_loader, device, num_classes, num_models):
    # train hierarchical model with dual outputs

    print("\n" + "=" * 60)
    print(f"TRAINING: {model_name.upper()}")
    print("=" * 60)

    # get config for this model (loss_type is 'hierarchical' for this model)
    config = get_config_for_model(model_name, loss_type, num_classes, num_models)

    # create model
    print(f"\ncreating model: {model_name}")
    model = get_model(model_name, config['num_classes'], num_models=num_models)
    model = model.to(device)

    # print model info
    params = count_parameters(model)
    print(f"total parameters: {params['total']:,}")
    print(f"trainable parameters: {params['trainable']:,}")

    # hierarchical loss function
    criterion = HierarchicalLoss(alpha=HIERARCHICAL_ALPHA)

    # optimizer
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )

    # scheduler (use make accuracy for scheduling)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=config['scheduler_factor'],
        patience=config['scheduler_patience']
    )

    # mixed precision scaler
    scaler = GradScaler('cuda')

    # early stopping (based on make accuracy)
    early_stopping = EarlyStopping(patience=config['early_stopping_patience'])

    # training history (custom for hierarchical)
    history_data = {}
    history_data['train_loss'] = []
    history_data['train_make_acc'] = []
    history_data['train_model_acc'] = []
    history_data['val_loss'] = []
    history_data['val_make_acc'] = []
    history_data['val_model_acc'] = []
    history_data['lr'] = []

    # best model tracking
    best_val_make_acc = 0.0

    # save config
    config_path = CHECKPOINT_DIR / f'{model_name}_config.json'
    save_config(config, config_path)
    print(f"saved config to: {config_path}")

    # training loop
    print(f"\nstarting training for {config['num_epochs']} epochs...")
    print(f"batch size: {config['batch_size']}")
    print(f"learning rate: {config['learning_rate']}")
    print(f"hierarchical alpha: {HIERARCHICAL_ALPHA}")
    print("-" * 50)

    start_time = datetime.now()

    for epoch in range(config['num_epochs']):
        print(f"\nepoch {epoch + 1}/{config['num_epochs']}")

        # get current learning rate
        current_lr = optimizer.param_groups[0]['lr']

        # train
        train_loss, train_make_acc, train_model_acc = train_hierarchical_epoch(
            model, train_loader, optimizer, criterion, device, scaler, config['use_amp']
        )

        # validate
        val_loss, val_make_acc, val_model_acc = validate_hierarchical(
            model, val_loader, criterion, device
        )

        # update history
        history_data['train_loss'].append(train_loss)
        history_data['train_make_acc'].append(train_make_acc)
        history_data['train_model_acc'].append(train_model_acc)
        history_data['val_loss'].append(val_loss)
        history_data['val_make_acc'].append(val_make_acc)
        history_data['val_model_acc'].append(val_model_acc)
        history_data['lr'].append(current_lr)

        # print epoch summary
        print(f"train loss: {train_loss:.4f}")
        print(f"train make acc: {train_make_acc:.2f}% | train model acc: {train_model_acc:.2f}%")
        print(f"val make acc: {val_make_acc:.2f}% | val model acc: {val_model_acc:.2f}%")
        print(f"learning rate: {current_lr:.6f}")

        # update scheduler (based on make accuracy)
        scheduler.step(val_make_acc)

        # save best model (based on make accuracy)
        if val_make_acc > best_val_make_acc:
            best_val_make_acc = val_make_acc
            checkpoint_path = CHECKPOINT_DIR / f'{model_name}_best.pth'
            save_checkpoint(model, optimizer, epoch, val_make_acc, checkpoint_path)
            print(f"saved best model (val_make_acc: {val_make_acc:.2f}%)")

        # check early stopping
        if early_stopping(val_make_acc):
            print(f"\nearly stopping triggered at epoch {epoch + 1}")
            break

    end_time = datetime.now()
    training_time = end_time - start_time

    # save training history
    history_path = CHECKPOINT_DIR / f'{model_name}_history.json'
    with open(history_path, 'w') as f:
        json.dump(history_data, f, indent=2)
    print(f"\nsaved history to: {history_path}")

    # plot hierarchical training curves
    plot_path = RESULTS_DIR / f'{model_name}_training_curves.png'
    plot_hierarchical_curves(history_data, save_path=plot_path)

    # save summary
    summary = {}
    summary['model_name'] = model_name
    summary['total_parameters'] = params['total']
    summary['trainable_parameters'] = params['trainable']
    summary['epochs_trained'] = len(history_data['train_loss'])
    summary['best_val_make_accuracy'] = best_val_make_acc
    summary['final_train_make_accuracy'] = history_data['train_make_acc'][-1]
    summary['final_train_model_accuracy'] = history_data['train_model_acc'][-1]
    summary['final_val_make_accuracy'] = history_data['val_make_acc'][-1]
    summary['final_val_model_accuracy'] = history_data['val_model_acc'][-1]
    summary['training_time'] = str(training_time)
    summary['checkpoint_path'] = str(CHECKPOINT_DIR / f'{model_name}_best.pth')

    summary_path = CHECKPOINT_DIR / f'{model_name}_summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"saved summary to: {summary_path}")

    # print final summary
    print("\n" + "-" * 50)
    print(f"{model_name.upper()} COMPLETE")
    print(f"best val make accuracy: {best_val_make_acc:.2f}%")
    print(f"final val model accuracy: {history_data['val_model_acc'][-1]:.2f}%")
    print(f"training time: {training_time}")
    print("-" * 50)

    return best_val_make_acc, training_time

## 6. Initialize Training Environment

In [12]:
# set random seed
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# create directories
CHECKPOINT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"gpu: {gpu_name}")

using device: cuda
gpu: NVIDIA GeForce RTX 3060


## 7. Load Datasets

In [13]:
# load data using pre-processed images (224x224, bbox-cropped)
print("loading datasets from pre-processed images...")

train_transform = get_processed_train_transforms()
val_transform = get_processed_val_transforms()

# create train dataset
train_dataset = ProcessedCompCarsDataset(
    root_dir=PROCESSED_DATA_ROOT,
    split='train',
    transform=train_transform,
    task=TASK
)

# create val dataset
val_dataset = ProcessedCompCarsDataset(
    root_dir=PROCESSED_DATA_ROOT,
    split='val',
    transform=val_transform,
    task=TASK
)

# get class counts from dataset
NUM_CLASSES = train_dataset.num_makes
NUM_MODELS = train_dataset.num_models

print(f"training set: {len(train_dataset)} images")
print(f"validation set: {len(val_dataset)} images")
print(f"number of make classes: {NUM_CLASSES}")
print(f"number of model classes: {NUM_MODELS}")

loading datasets from pre-processed images...
training set: 14414 images
validation set: 1602 images
number of make classes: 75
number of model classes: 431


In [14]:
# create dataloaders
print("creating dataloaders...")

train_loader = get_dataloader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    drop_last=True
)

val_loader = get_dataloader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    drop_last=False
)

print(f"training batches: {len(train_loader)}")
print(f"validation batches: {len(val_loader)}")

creating dataloaders...
training batches: 450
validation batches: 51


In [15]:
# check if hierarchical model is in experiments
experiment_models = [exp[0] for exp in EXPERIMENTS]
has_hierarchical = 'hierarchical' in experiment_models

# create hierarchical dataloaders if needed
hierarchical_train_loader = None
hierarchical_val_loader = None

if has_hierarchical:
    print("\ncreating hierarchical dataloaders...")

    hierarchical_train_dataset = ProcessedCompCarsDataset(
        root_dir=PROCESSED_DATA_ROOT,
        split='train',
        transform=train_transform,
        task='hierarchical'
    )

    hierarchical_val_dataset = ProcessedCompCarsDataset(
        root_dir=PROCESSED_DATA_ROOT,
        split='val',
        transform=val_transform,
        task='hierarchical'
    )

    hierarchical_train_loader = get_dataloader(
        hierarchical_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        drop_last=True
    )

    hierarchical_val_loader = get_dataloader(
        hierarchical_val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        drop_last=False
    )

    print(f"hierarchical training batches: {len(hierarchical_train_loader)}")
    print(f"hierarchical validation batches: {len(hierarchical_val_loader)}")


creating hierarchical dataloaders...
hierarchical training batches: 450
hierarchical validation batches: 51


## 8. Run Training Experiments

In [ ]:

results = []
total_experiments = len(EXPERIMENTS)

for exp_idx, (model_name, loss_type) in enumerate(EXPERIMENTS):
    print(f"\n{'#' * 70}")
    print(f"# EXPERIMENT {exp_idx + 1}/{total_experiments}: {model_name} + {loss_type}")
    print(f"{'#' * 70}")

    if model_name == 'hierarchical':
        best_acc, train_time = train_hierarchical_model(
            model_name,
            loss_type,
            hierarchical_train_loader,
            hierarchical_val_loader,
            device,
            num_classes=NUM_CLASSES,
            num_models=NUM_MODELS
        )
        experiment_name = model_name
    else:
        best_acc, train_time = train_model(
            model_name,
            loss_type,
            train_loader,
            val_loader,
            device,
            num_classes=NUM_CLASSES,
            num_models=NUM_MODELS
        )
        experiment_name = f"{model_name}_{loss_type}"

    result = {}
    result['experiment'] = experiment_name
    result['model'] = model_name
    result['loss_type'] = loss_type
    result['best_val_accuracy'] = best_acc
    result['training_time'] = str(train_time)
    results.append(result)

## 9. Training Summary

In [17]:
# print final summary of all models
print("\n" + "=" * 60)
print("ALL TRAINING COMPLETE")
print("=" * 60)
print(f"\n{'experiment':<30} {'best val acc':>15} {'time':>20}")
print("-" * 65)

for result in results:
    experiment = result['experiment']
    acc = result['best_val_accuracy']
    time = result['training_time']
    print(f"{experiment:<30} {acc:>14.2f}% {time:>20}")

print("-" * 55)


ALL TRAINING COMPLETE

experiment                        best val acc                 time
-----------------------------------------------------------------
resnet50_label_smoothing                97.57%       1:23:22.337685
resnet50_ce                             97.75%       1:21:39.099510
resnet50_focal                          96.69%       1:21:48.780356
simplecnn_label_smoothing               74.78%       0:22:40.840144
efficientnet_label_smoothing            97.75%       0:52:42.147487
resnet50_se_label_smoothing             98.00%       1:25:33.489777
hierarchical                            98.94%       1:24:28.289278
-------------------------------------------------------
